# Tasks - Genomics Part 2: From alignments to variants

<div class="alert alert-info" style="background-color:rgba(212, 240, 255, 0.5); border-color:rgba(212, 240, 255, 1)">
Use this notebook as a template to solve the following tasks and to run the variant calling pipeline. 

- Use Markdown cells for headings and explanations
- Use code cells to run bash commands with the `!` prefix or the `%%bash` magic command. **Don't forget to actually run the commands by pressing **Shift+Enter** or **Ctrl+Enter**!**
- **Some code cells can be run as they are without any modification, but when a task asks you to write the relevant code, make sure to include it in the prepared code cell below.**
- Provide explanations and comments in the code cells by starting them with `#`
- For screenshot tasks, create a screenshot, upload it to the `practical3` folder and insert the image in the Markdown cell

</div>

## Connecting to last week

### Checking that all the necessary files are present

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3/

# List all BAM files
ls -lh results/bam_processed/*_sorted_dedup.bam

## Preparations

### Reference genome dictionary

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create sequence dictionary
picard CreateSequenceDictionary \
  R=data/reference/chr17.fa \
  O=data/reference/chr17.dict

Checking to see if it was created:

In [ ]:
%%bash

cd genomics_practical3

# Verify the dictionary was created
ls -lh data/reference/chr17.*

### Validating alignment files

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Validate alignment files
picard ValidateSamFile \
  I=results/bam_processed/P1_AD_TP53_sorted_dedup.bam \
  MODE=SUMMARY

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 1

Run the validation command for all BAM files using a `for` loop and check for errors and warnings. Do you see any problems? If so, how can you fix them?

</div>

In [ ]:
%%bash

# write your code here

**Your answer:**

## Germline variant calling with GATK HaplotypeCaller

### Step 1: Creating GVCFs for each sample

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create directory for GVCFs
mkdir -p results/gvcf

# Run HaplotypeCaller on P1 normal sample
gatk HaplotypeCaller \
  -R data/reference/chr17.fa \
  -I results/bam_processed/P1_NAT_TP53_sorted_dedup.bam \
  -O results/gvcf/P1_NAT_TP53.g.vcf.gz \
  -ERC GVCF \
  -L 17:7661779-7687550

# -R = reference genome
# -I = input BAM file
# -O = output GVCF
# -ERC GVCF = this option is needed for joint calling later
# -L = limit to TP53 region

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 2

Run the above command for all normal samples using a `for` loop and check that the output files are created.

1. How long does it take to run for all four normal samples?
2. Why do we use `-L` to limit to the TP53 region?
3. How would runtime change for whole genome calling?

</div>

In [ ]:
%%bash

# write your code here

**Your answers:**

1. Runtime:
2. Why use `-L`?
3. Runtime for WGS:

### Step 2: Joint genotyping

#### Creating the database

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Combine GVCFs into a GenomicsDB
mkdir -p results/genomicsdb

gatk GenomicsDBImport \
  -V results/gvcf/P1_NAT_TP53.g.vcf.gz \
  -V results/gvcf/P2_NAT_TP53.g.vcf.gz \
  -V results/gvcf/P3_NAT_TP53.g.vcf.gz \
  -V results/gvcf/P4_NAT_TP53.g.vcf.gz \
  --genomicsdb-workspace-path results/genomicsdb/tp53_database \
  -L 17:7661779-7687550

#### Joint genotyping

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create final germline VCF
mkdir -p results/vcf/germline

gatk GenotypeGVCFs \
  -R data/reference/chr17.fa \
  -V gendb://results/genomicsdb/tp53_database \
  -O results/vcf/germline/germline_all_samples.vcf.gz

## Understanding the VCF file

### Examining the VCF file

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# View first 20 lines of header
zcat results/vcf/germline/germline_all_samples.vcf.gz | grep "^##" | head -20
echo "----------------------------------------------------------------"

# View column names
zcat results/vcf/germline/germline_all_samples.vcf.gz | grep "^#CHROM"
echo "----------------------------------------------------------------"

# View first 5 variants
zcat results/vcf/germline/germline_all_samples.vcf.gz | grep -v "^#" | head -5

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 3

Find the variant at position chr17:7,676,325 (TP53 coding region) and print out the data line.

1. What kind of a mutation is this? What is the reference allele and the variant allele?
2. What is the genotype of each sample at this position?
3. What is the lowest coverage at this position and for which sample?
4. Based on the data, would you say that the reference genome represents the investigated population of our 4 patients well? Why or why not?

</div>

In [ ]:
%%bash

# write your code here

**Your answers:**

### Visualizing germline variants

In [ ]:
# importing plotting packages
import matplotlib.pyplot as plt
import seaborn as sns
# importing numpy for matrix handling
import numpy as np
# importing pyvcf for vcf file parsing
import vcf

# TP53 gene location on chr17
TP53_min = 7661779
TP53_max = 7687550

# sample names
sample_names = ["P1_Normal", "P2_Normal", "P3_Normal", "P4_Normal"]

# genotype dictionary
gt_dict = {'0/0': 1, '0/1': 2, '0|1': 2, '0/2': 2, '0/3': 2, '1/1': 3, '1|1': 3}

# First pass: collect all variant positions
variant_positions = []
vcf_reader = vcf.Reader(filename='genomics_practical3/results/vcf/germline/germline_all_samples.vcf.gz')
for record in vcf_reader:
    if TP53_min <= record.POS <= TP53_max:
        variant_positions.append(record.POS)

# Create data matrix with only variant positions
n_variants = len(variant_positions)
data_matrix = np.empty((len(sample_names), n_variants))
data_matrix[:] = np.nan

# Create position-to-index mapping
pos_to_idx = {pos: idx for idx, pos in enumerate(variant_positions)}

# Second pass: fill the matrix
vcf_reader = vcf.Reader(filename='genomics_practical3/results/vcf/germline/germline_all_samples.vcf.gz')
for record in vcf_reader:
    if record.POS in pos_to_idx:
        idx = pos_to_idx[record.POS]
        for sID, s in enumerate(sample_names):
            data_matrix[sID][idx] = gt_dict.get(record.genotype(name=s)['GT'], np.nan)

# Prepare colormap
from matplotlib.colors import LinearSegmentedColormap
colors = ['#ffe6a7', '#76c893', '#168aad']
cmap = LinearSegmentedColormap.from_list('my_cmap', colors, N=3)
plt.rcParams['font.family'] = 'Lato'  # or 'DejaVu Sans', 'Helvetica', etc.

# Plotting
grid_kws = {"height_ratios": (.9, .05), "hspace": .3}
f, (ax, cbar_ax) = plt.subplots(2, gridspec_kw=grid_kws, figsize=(15, 3))

ax = sns.heatmap(data_matrix, ax=ax,
                 cbar_ax=cbar_ax,
                 xticklabels=['chr17:' + '{:,}'.format(p) for p in variant_positions],  # Show actual positions formatted nicely
                 yticklabels=sample_names,
                 cbar_kws={"orientation": "horizontal", 'ticks': [1+1/3, 2, 3-1/3]},
                 cmap=cmap,
                 linewidths=0.5,  # Add gridlines between variants
                 linecolor='white')

cbar_ax.set_xticklabels(['homozygous reference', 'heterozygous', 'homozygous variant'], fontsize=14) # Set colorbar labels

# Adjustments
cbar_ax.tick_params(size=0)
ax.xaxis.tick_top() # Move x-axis labels to top
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=8) # Rotate x-axis labels for readability
ax.tick_params(left=False, top=False)  # Remove tick marks on both axes

plt.show()

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 4

Look at the figure above in detail and answer the following questions.

1. In position 7,674,892, one of the alleles of patient 4 carries a C base instead of the reference T, so this is a heterozygous mutation. None of the other patients seem to have this mutation, based on the figure. Let's search for this variant in the [dbSNP database](https://www.ncbi.nlm.nih.gov/snp/), which contains known variants and their frequency and potential effects. This variant has the rsID [rs1800372](https://www.ncbi.nlm.nih.gov/snp/rs1800372). Looking at the summary page, answer the following questions:
    - What is the type of the variant?
    - The consequence of the variant is listed as "Synonymous Variant". What does this mean?
    - What is the frequency of this variant in the European population?
    - How many studies mentioned this variant?
    - What are the genomic coordinates of this variant in the GRCh37 version of the human reference genome? Why is this different from the coordinates in the VCF file?
2. Look at the figure above and find those positions where all four patients have a germline mutation (either homozygous or heterozygous). How many such positions are there? Check the dbSNP database for these positions and find if any of these variants have a "pathogenic" clinical significance (i.e. it is known to cause a disease).

</div>

**Your answers:**

---

## Somatic variant calling with Mutect2

### Step 1: Creating a Panel of Normals (PoN)

#### Run Mutect2 in tumor-only mode on normal samples

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create directory for PoN VCFs
mkdir -p results/pon_vcfs

# Run Mutect2 in tumor-only mode for each NAT sample
for PATIENT in 1 2 3 4; do
  echo "Processing P${PATIENT}_NAT for PoN..."
  
  gatk Mutect2 \
    -R data/reference/chr17.fa \
    -I results/bam_processed/P${PATIENT}_NAT_TP53_sorted_dedup.bam \
    -tumor P${PATIENT}_Normal \
    -L 17:7661779-7687550 \
    --max-mnp-distance 0 \
    -O results/pon_vcfs/P${PATIENT}_NAT_for_pon.vcf.gz
    
  echo "P${PATIENT}_NAT complete!"
done

# Parameters:
# -tumor = sample name from read group
# --max-mnp-distance 0 = treat MNPs as separate events (recommended for PoN)
# NO -normal parameter = tumor-only mode

#### Combine the normal sample VCFs into a PoN

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create database of PoN VCFs
gatk GenomicsDBImport \
  -R data/reference/chr17.fa \
  -L 17:7661779-7687550 \
  --genomicsdb-workspace-path results/genomicsdb/pon_database \
  -V results/pon_vcfs/P1_NAT_for_pon.vcf.gz \
  -V results/pon_vcfs/P2_NAT_for_pon.vcf.gz \
  -V results/pon_vcfs/P3_NAT_for_pon.vcf.gz \
  -V results/pon_vcfs/P4_NAT_for_pon.vcf.gz \


# Create PoN from the database
gatk CreateSomaticPanelOfNormals \
  -R data/reference/chr17.fa \
  -V gendb://results/genomicsdb/pon_database \
  --germline-resource /shared/OMICS/practical_04/resource/af-only-gnomad_grch38_17.vcf.gz \
  -O results/pon_vcfs/pon.vcf.gz

### Step 2: Calling somatic variants with Mutect2 (with PoN)

Checking if sample names match:

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Check the read group of the BAM file
samtools view -H results/bam_processed/P1_AD_TP53_sorted_dedup.bam | grep "^@RG"

Running Mutect2 in matched tumor-normal mode:

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create directory for somatic VCFs
mkdir -p results/vcf/somatic
mkdir -p results/f1r2

# Run Mutect2 for P1 with PoN (tumor = AD, normal = NAT)
gatk Mutect2 \
  -R data/reference/chr17.fa \
  -I results/bam_processed/P1_AD_TP53_sorted_dedup.bam \
  -I results/bam_processed/P1_NAT_TP53_sorted_dedup.bam \
  -tumor P1_Adenoma \
  -normal P1_Normal \
  -pon results/pon_vcfs/pon.vcf.gz \
  --germline-resource /shared/OMICS/practical_04/resource/af-only-gnomad_grch38_17.vcf.gz \
  --f1r2-tar-gz results/f1r2/P1_f1r2.tar.gz \
  -L 17:7661779-7687550 \
  -O results/vcf/somatic/P1_somatic_raw.vcf.gz

# Parameters:
# -I = input BAMs (tumor first, then normal)
# -tumor = tumor sample name (from read group)
# -normal = normal sample name
# -pon = Panel of Normals (filters technical artifacts)
# -L = region to call (TP53)
# -O = output VCF

### Step 3: Estimating read orientation bias

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Estimate read orientation bias
gatk LearnReadOrientationModel \
  -I results/f1r2/P1_f1r2.tar.gz \
  -O results/f1r2/P1_read_orientation_model.tar.gz

### Step 4: Estimating contamination

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create directory for contamination Metrics
mkdir -p results/contamination_metrics

gatk GetPileupSummaries \
-I results/bam_processed/P1_AD_TP53_sorted_dedup.bam \
-V /shared/OMICS/practical_04/resource/small_exac_common_3_grch38_17.vcf.gz \
-L 17:7661779-7687550 \
-O results/contamination_metrics/P1_getpileupsummaries.table

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Estimate contamination
gatk CalculateContamination \
-I results/contamination_metrics/P1_getpileupsummaries.table \
-tumor-segmentation results/contamination_metrics/P1_segments.table \
-O results/contamination_metrics/P1_contamination.table

### Step 5: Filtering somatic variants

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Filter somatic variants
gatk FilterMutectCalls \
  -R data/reference/chr17.fa \
  -V results/vcf/somatic/P1_somatic_raw.vcf.gz \
  --contamination-table results/contamination_metrics/P1_contamination.table \
  --stats results/vcf/somatic/P1_somatic_raw.vcf.gz.stats \
  --tumor-segmentation results/contamination_metrics/P1_segments.table \
  --ob-priors results/f1r2/P1_read_orientation_model.tar.gz \
  -O results/vcf/somatic/P1_somatic_filtered.vcf.gz

### Processing all patients

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Create necessary directories
mkdir -p results/vcf/somatic
mkdir -p results/f1r2
mkdir -p results/contamination_metrics

# Process all tumor-normal pairs with PoN
for PATIENT in 1 2 3 4; do
  
  # Determine tumor type (AD or CRC)
  if [ $PATIENT -le 2 ]; then # patients 1 and 2 are adenomas, patients 3 and 4 are CRCs
    TUMOR="P${PATIENT}_AD"
    TUMOR_NAME="P${PATIENT}_Adenoma"
  else
    TUMOR="P${PATIENT}_CRC"
    TUMOR_NAME="P${PATIENT}_Cancer"
  fi
  
  NORMAL="P${PATIENT}_NAT"
  NORMAL_NAME="P${PATIENT}_Normal"
  
  echo "Processing Patient $PATIENT ($TUMOR vs $NORMAL)..."
  
  # Call somatic variants with PoN
  gatk Mutect2 \
    -R data/reference/chr17.fa \
    -I results/bam_processed/${TUMOR}_TP53_sorted_dedup.bam \
    -I results/bam_processed/${NORMAL}_TP53_sorted_dedup.bam \
    -tumor $TUMOR_NAME \
    -normal $NORMAL_NAME \
    -pon results/pon_vcfs/pon.vcf.gz \
    --germline-resource /shared/OMICS/practical_04/resource/af-only-gnomad_grch38_17.vcf.gz \
    --f1r2-tar-gz results/f1r2/P${PATIENT}_f1r2.tar.gz \
    -L 17:7661779-7687550 \
    -O results/vcf/somatic/P${PATIENT}_somatic_raw.vcf.gz    
  
  # Estimate read orientation bias
  gatk LearnReadOrientationModel \
    -I results/f1r2/P${PATIENT}_f1r2.tar.gz \
    -O results/f1r2/P${PATIENT}_read_orientation_model.tar.gz

  # Estimate contamination
  gatk GetPileupSummaries \
    -I results/bam_processed/${TUMOR}_TP53_sorted_dedup.bam \
    -V /shared/OMICS/practical_04/resource/small_exac_common_3_grch38_17.vcf.gz \
    -L 17:7661779-7687550 \
    -O results/contamination_metrics/P${PATIENT}_getpileupsummaries.table    

  gatk CalculateContamination \
    -I results/contamination_metrics/P${PATIENT}_getpileupsummaries.table \
    -tumor-segmentation results/contamination_metrics/P${PATIENT}_segments.table \
    -O results/contamination_metrics/P${PATIENT}_contamination.table    

  # Filter variants
  gatk FilterMutectCalls \
    -R data/reference/chr17.fa \
    -V results/vcf/somatic/P${PATIENT}_somatic_raw.vcf.gz \
    --contamination-table results/contamination_metrics/P${PATIENT}_contamination.table \
    --stats results/vcf/somatic/P${PATIENT}_somatic_raw.vcf.gz.stats \
    --tumor-segmentation results/contamination_metrics/P${PATIENT}_segments.table \
    --ob-priors results/f1r2/P${PATIENT}_read_orientation_model.tar.gz \
    -O results/vcf/somatic/P${PATIENT}_somatic_filtered.vcf.gz
  
  echo "Patient $PATIENT complete!"
done

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 5

Run the somatic variant calling pipeline for all patients. Then count the number of raw somatic variants and those that passed all filtering steps for each patient. Search online for the mutations you find and discuss whether they can be considered oncogenic. What are their predicted functional consequences, were they found in cancer samples before? (E.g. try using the [COSMIC database](https://cancer.sanger.ac.uk/cosmic/), which summarizes mutations in various cancer samples.)

</div>

In [ ]:
%%bash

# write your code here

**Your answer:**

## Analyzing mutation patterns

Copying the prepared VCF file for patient P1:

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3

# Copy the file to your own folder
cp /shared/OMICS/practical_04/P1_somatic_filtered_chr17.vcf.gz results/vcf/somatic/P1_somatic_filtered_chr17.vcf.gz

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 6

How many raw somatic variants and those that passed all filtering steps were found in chromosome 17 for patient P1?

</div>

In [ ]:
%%bash

# write your code here

**Your answer:**

### Variant annotation

In [ ]:
%%bash

# Navigate to the project directory
cd genomics_practical3/

snpEff -Xmx8g GRCh38.99 \
  results/vcf/somatic/P1_somatic_filtered_chr17.vcf.gz \
  > results/vcf/somatic/P1_somatic_filtered_chr17_annotated.vcf

### Visualizing the results

#### Mutation types based on genomic characteristics

In [ ]:
import matplotlib.pyplot as plt
import vcf
from collections import Counter

# Set font type
plt.rcParams['font.family'] = 'Lato'

# Set file path
annotated_vcf = "genomics_practical3/results/vcf/somatic/P1_somatic_filtered_chr17_annotated.vcf"

# Parse VCF with pyvcf
vcf_reader = vcf.Reader(filename=annotated_vcf)

mutation_types = []
snp_changes = []

for record in vcf_reader:
    # Only PASS variants
    if record.FILTER and 'PASS' not in record.FILTER:
        continue
    
    ref = record.REF
    alt = str(record.ALT[0])  # Take first ALT allele
    
    # Classify mutation type
    if len(ref) == 1 and len(alt) == 1:
        mutation_types.append('SNP')
        snp_changes.append(f"{ref}>{alt}")
    elif len(ref) > len(alt):
        mutation_types.append('deletion')
    elif len(ref) < len(alt):
        mutation_types.append('insertion')
    else:
        mutation_types.append('complex')

# Count mutation types
type_counts = Counter(mutation_types)

# Define complementary pairs
complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

def get_canonical_change(change):
    """Convert to canonical form (always show pyrimidine>X)"""
    ref, alt = change.split('>')
    
    # Convert to pyrimidine reference (C or T)
    if ref in ['A', 'G']:
        # Get complement of both
        ref = complement[ref]
        alt = complement[alt]
    
    return f"{ref}>{alt}"

# Group SNP changes
grouped_snp_changes = Counter()
for change in snp_changes:
    canonical = get_canonical_change(change)
    grouped_snp_changes[canonical] += 1

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

# Subplot 1: Mutation types
colors_types = ['#3498DB', '#E74C3C', '#F39C12', '#9B59B6']
ax1.bar(type_counts.keys(), type_counts.values(), color=colors_types[:len(type_counts)], 
        edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Mutation type', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of mutations', fontsize=12, fontweight='bold')
ax1.set_title('Distribution of mutation types', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Add counts on bars
for i, (mut_type, count) in enumerate(type_counts.items()):
    ax1.text(i, count + 0.5, str(count), ha='center', fontweight='bold')

# Subplot 2: Grouped SNP base changes
if grouped_snp_changes:
    sorted_snps = dict(sorted(grouped_snp_changes.items(), key=lambda x: x[1], reverse=True))
    
    transitions = ['C>T', 'T>C']  # Canonical transitions
    colors_snps = ['#2ECC71' if snp in transitions else '#E67E22' 
                   for snp in sorted_snps.keys()]
    
    ax2.bar(sorted_snps.keys(), sorted_snps.values(), color=colors_snps, 
            edgecolor='black', linewidth=1.5)
    ax2.set_xlabel('Base change (grouped by strand)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Number of SNPs', fontsize=12, fontweight='bold')
    ax2.set_title('SNP base change distribution\n(C>A includes G>T, etc.)', 
                  fontsize=14, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add counts on bars
    for i, (change, count) in enumerate(sorted_snps.items()):
        ax2.text(i, count + 0.2, str(count), ha='center', fontweight='bold')
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2ECC71', edgecolor='black', label='Transition (C↔T)'),
        Patch(facecolor='#E67E22', edgecolor='black', label='Transversion')
    ]
    ax2.legend(handles=legend_elements, loc='upper right')
else:
    ax2.text(0.5, 0.5, 'No SNPs found', ha='center', va='center', 
             transform=ax2.transAxes, fontsize=14)
    ax2.axis('off')

ax1.set_ylim(0,35) # Set vertical axis ranges
ax2.set_ylim(0,20)
plt.show()

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 7

Based on the figure, there are one somatic insertion and two complex alterations in the adenoma sample of patient P1. Which ones are these? (Find their genomic position and the reference and variant alleles.)

</div>

In [ ]:
# write your code here (Hint: the start of the code generating the figure above can serve as a good base for this task.)

**Your answer:**

#### Mutation impact and consequence

In [ ]:
import matplotlib.pyplot as plt
import vcf
from collections import Counter

# Set font type
plt.rcParams['font.family'] = 'Lato'

# Set file path
annotated_vcf = "genomics_practical3/results/vcf/somatic/P1_somatic_filtered_chr17_annotated.vcf"

# Parse VCF with pyvcf
vcf_reader = vcf.Reader(filename=annotated_vcf)

impacts = []
consequences = []
genes = []

for record in vcf_reader:
    # Only PASS variants
    if record.FILTER and 'PASS' not in record.FILTER:
        continue
    
    # Extract ANN field (SnpEff annotation)
    if hasattr(record, 'INFO') and 'ANN' in record.INFO:
        ann_list = record.INFO['ANN']
        
        # Take first annotation
        ann = ann_list[0] if isinstance(ann_list, list) else ann_list
        
        # Format: Allele|Annotation|Impact|Gene|Gene_ID|Feature_Type|...
        parts = ann.split('|')
        
        if len(parts) >= 4:
            consequence = parts[1]
            impact = parts[2]
            gene = parts[3]
            
            consequences.append(consequence)
            impacts.append(impact)
            if gene:
                genes.append(gene)

# Count occurrences
impact_counts = Counter(impacts)
consequence_counts = Counter(consequences)
gene_counts = Counter(genes)

# Create figure with subplots
fig = plt.figure(figsize=(15, 5))
gs = fig.add_gridspec(1, 3, hspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])

impact_colors = {
    'HIGH': '#E74C3C',
    'MODERATE': '#F39C12',
    'LOW': '#3498DB',
    'MODIFIER': '#95A5A6'
}
sorted_impacts = sorted(impact_counts.items(), 
                       key=lambda x: ['HIGH', 'MODERATE', 'LOW', 'MODIFIER'].index(x[0]) 
                       if x[0] in ['HIGH', 'MODERATE', 'LOW', 'MODIFIER'] else 999)

labels = [imp for imp, _ in sorted_impacts]
sizes = [cnt for _, cnt in sorted_impacts]
colors = [impact_colors.get(imp, '#95A5A6') for imp in labels]

# Create pie chart with counts
wedges, texts, autotexts = ax1.pie(sizes, labels=labels, colors=colors, autopct='%d%%',
                                     startangle=90, counterclock=False,
                                     wedgeprops={'edgecolor': 'black', 'linewidth': 1.5},
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})

# Make percentage text white and bold
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(12)

ax1.set_title('Mutation impact distribution', fontsize=14, fontweight='bold')

top_consequences = dict(consequence_counts.most_common(8))
ax2.bar(list(top_consequences.keys()), list(top_consequences.values()),
        color='#3498DB', edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Number of mutations', fontsize=12, fontweight='bold')
ax2.set_xlabel('Consequence type', fontsize=12, fontweight='bold')
ax2.set_title('Top variant consequences', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)  # Rotate x-axis labels
ax2.set_xticks(range(len(top_consequences)))
ax2.set_xticklabels(top_consequences.keys(), ha='right') 
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.set_ylim(0,10)

# Add counts on top of bars
for i, (cons, cnt) in enumerate(top_consequences.items()):
    ax2.text(i, cnt + 0.3, str(cnt), ha='center', fontweight='bold')

top_genes = {k:v for k,v in gene_counts.items() if v > 1}
colors_genes = plt.cm.Set3(range(len(top_genes)))
ax3.bar(top_genes.keys(), top_genes.values(), 
        color=colors_genes, edgecolor='black', linewidth=1.5)
ax3.set_xlabel('Gene', fontsize=12, fontweight='bold')
ax3.set_ylabel('Number of mutations', fontsize=12, fontweight='bold')
ax3.set_title('Genes with multiple mutations', fontsize=14, fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
ax3.set_xticks(range(len(top_genes)))
ax3.set_xticklabels(top_genes.keys(), ha='right') 
ax3.grid(axis='y', alpha=0.3, linestyle='--')
ax3.set_ylim(0,4)

# Add counts
for i, (gene, cnt) in enumerate(top_genes.items()):
    ax3.text(i, cnt + 0.3, str(cnt), ha='center', fontweight='bold')

plt.show()

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 8

The figure shows that about 6% of the somatic mutations have HIGH impact. How many such mutations were found and in which genes are they located?

</div>

In [ ]:
# write your code here (Hint: the start of the code generating the figure above can serve as a good base for this task.)

**Your answer:**

#### Mutation distribution along the chromosome and variant allele frequencies

In [ ]:
import matplotlib.pyplot as plt
import vcf

# Set font type
plt.rcParams['font.family'] = 'Lato'

# Set file path
annotated_vcf = "genomics_practical3/results/vcf/somatic/P1_somatic_filtered_chr17_annotated.vcf"

# Parse VCF with pyvcf
vcf_reader = vcf.Reader(filename=annotated_vcf)

mutations = []

for record in vcf_reader:
    # Only PASS variants
    if record.FILTER and 'PASS' not in record.FILTER:
        continue
    
    pos = record.POS
    ref = record.REF
    alt = str(record.ALT[0])
    
    impact = 'MODIFIER'
    
    # Extract annotation for impact
    if hasattr(record, 'INFO') and 'ANN' in record.INFO:
        ann_list = record.INFO['ANN']
        ann = ann_list[0] if isinstance(ann_list, list) else ann_list
        parts = ann.split('|')
        
        if len(parts) >= 3:
            impact = parts[2]
    
    # The adenome sample is the first one in the VCF file
    tumor_sample = record.samples[0]
    
    # Try to get AF from different possible fields
    af = None
    if hasattr(tumor_sample.data, 'AF') and tumor_sample.data.AF:
        af = tumor_sample.data.AF[0] if isinstance(tumor_sample.data.AF, list) else tumor_sample.data.AF
    elif hasattr(tumor_sample.data, 'AD') and tumor_sample.data.AD:
        # Calculate AF from allele depths (AD)
        ad = tumor_sample.data.AD
        if len(ad) >= 2 and sum(ad) > 0:
            af = ad[1] / sum(ad)  # alt / (ref + alt)
    
    # If no AF found, skip this variant
    if af is None:
        continue
    
    mutations.append({
        'pos': pos,
        'ref': ref,
        'alt': alt,
        'impact': impact,
        'af': af
    })

# Sort by position
mutations = sorted(mutations, key=lambda x: x['pos'])

# Create figure
fig, ax = plt.subplots(figsize=(16, 3))

# Color by impact
impact_colors = {
    'HIGH': '#E74C3C',
    'MODERATE': '#F39C12',
    'LOW': '#3498DB',
    'MODIFIER': '#95A5A6'
}

# Plot lollipops with height = AF
for mut in mutations:
    pos = mut['pos']
    af = mut['af']
    impact = mut['impact']
    color = impact_colors.get(impact, '#95A5A6')
    
    # Stem (height = AF)
    ax.plot([pos, pos], [0, af], color=color, linewidth=2.5, alpha=0.8, zorder=2)
    
    # Lollipop head
    ax.scatter(pos, af, color=color, s=120, edgecolor='black', 
              linewidth=1.5, zorder=3, alpha=0.9)

# Formatting
ax.set_xlabel('Genomic position (on chr17)', fontsize=14, fontweight='bold')
ax.set_ylabel('Variant Allele Frequency (VAF)', fontsize=14, fontweight='bold')
ax.set_title('Somatic mutations across chromosome 17\nP1 adenoma sample', 
            fontsize=16, fontweight='bold', pad=20)
ax.set_ylim(0, 1.05)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.grid(axis='both', alpha=0.3, linestyle='--', zorder=1)

# Format x-axis with commas
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E74C3C', edgecolor='black', label='HIGH impact'),
    Patch(facecolor='#F39C12', edgecolor='black', label='MODERATE impact'),
    Patch(facecolor='#3498DB', edgecolor='black', label='LOW impact'),
    Patch(facecolor='#95A5A6', edgecolor='black', label='MODIFIER')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11,
         title='Mutation impact', title_fontsize=12)

plt.show()

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 9

Let’s assume that the adenoma sample was not contaminated with normal tissue at all. Looking at the figure, what can explain the relatively low (< 0.5) variant allele frequencies for the mutations?

</div>

**Your answer:**

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 10

Use the annotated VCF file to create a figure showing some interesting patterns in the data. Feel free to use any of the plots you created above as a starting point. Either run the code in a Code cell of your jupyter notebook or use your own code to create the figure and insert it into a Markdown cell.

</div>

In [ ]:
# your code for generating the figure

**Your figure:**

<!-- Upload your screenshot to the practical3 folder and use markdown to insert it:
![Figure](figure.png)
-->

---

## Congratulations!

You've completed all the tasks for Genomics Part 2 practical.

**Before submitting:**
1. Make sure all code cells have been run (Shift+Enter)
2. Check that all your answers are filled in
3. Save this notebook (Ctrl+S or Cmd+S)
4. Make sure this notebook is in your `practical3` folder
5. Name it: `firstname_lastname_practical4.ipynb`

**To download your notebook:**
- File → Download as → Notebook (.ipynb)